# 🛰️ Nova Neural Fusion: Cloud Training
Este cuaderno entrena a Nova en la nube usando GPUs de Google.

### 🚀 Instrucciones Rápidas:
1. Ve a **Entorno de ejecución** > **Cambiar tipo de entorno** > **T4 GPU**.
2. Ejecuta todas las celdas en orden.

In [ ]:
# 1. Instalar herramientas de velocidad (Unsloth)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

# 2. Cargar el cerebro base de Nova
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.2-1b-instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

In [ ]:
# 3. El Alma: Benedetti + Aristoteles
dataset_data = [
    {"instruction": "Dime una reflexion sobre el miedo.", "response": "Felicidad es ausencia de miedo, decia Benedetti. Vencerlo es ser libre."},
    {"instruction": "Escribe un poema breve.", "response": "Tengo miedo de verte, necesidad de verte... estoy jodido y radiante."},
    {"instruction": "Que es la virtud?", "response": "Para Aristoteles, es el justo medio entre el exceso y el defecto."}
]
dataset = Dataset.from_list([{"text": f"Instruction: {d['instruction']}\nResponse: {d['response']}"} for d in dataset_data])

In [ ]:
# 4. Entrenamiento Relampago
model = FastLanguageModel.get_peft_model(model, r = 16, lora_alpha = 16, random_state = 3407)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        max_steps = 60,
        learning_rate = 2e-4,
        output_dir = "outputs",
        logging_steps = 1
    ),
)
trainer.train()

In [ ]:
# 5. Exportar y Descargar Automaticamente
model.save_pretrained_gguf("nova_ultra_brain", tokenizer, quantization_method = "q4_k_m")
from google.colab import files
files.download("nova_ultra_brain-unsloth.Q4_K_M.gguf")